# EM-FloatFormer Stage 5 & 6 — Google Colab 执行工作簿 (`4B Real-SDR -> 9B -> 27B Mixed-NF4`)

- **零 Modal GPU 费用**：Stage 4 全部 `Qwen3.5-4B` 检查点（含 **12/12 ALL-PASS** `stage4_cn_sft_u1_s202611.pt`）与 Pluto/CTTC 双缓存 `.npz` 已托管至 GitHub Release `stage4-4b-colab-artifacts`。
- **W4 定理 2 量化红线守护**：`27B Mixed-NF4` 仅对 SwiGLU MLP (`gate_proj / up_proj / down_proj`) 启用 NF4，**严格跳过 `linear_attn`（Gated-DeltaNet）、`self_attn`、`embed_tokens`、`lm_head`、所有 LayerNorm 与 `FP32 SpecCayley`**，加载后强制执行 `audit_mixed_nf4_layout` 审计。
- **推荐运行时**：`A100-80GB`（可跑通 `4B + 9B BF16 + 27B Mixed-NF4` 全套）；`A100-40GB`（可跑通 `4B + 9B BF16 + 27B Mixed-NF4`）；`L4-24GB`（可跑通 `4B BF16 + 9B Mixed-NF4`）；`T4-16GB`（可跑通 `4B Mixed-NF4`）。

In [ ]:
#@title 1. 检测 Colab GPU 显存档位并挂载 Google Drive 持久化目录
import os, torch
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gib = props.total_memory / (1024 ** 3)
    print(f"GPU: {props.name} | VRAM: {vram_gib:.2f} GiB | BF16 native: {props.major >= 8}")
else:
    vram_gib = 0.0
    print("WARNING: No CUDA GPU detected! Select Runtime -> Change runtime type -> GPU.")

DRIVE_ROOT = Path("/content/drive/MyDrive/EM-FloatFormer-Stage6")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = DRIVE_ROOT
    print(f"Google Drive mounted. Persistent dir: {WORK_ROOT}")
except Exception as exc:
    WORK_ROOT = Path("/content/EM-FloatFormer-Stage6")
    print(f"Drive mount skipped ({exc}); using local runtime dir: {WORK_ROOT}")

WORK_DIR = WORK_ROOT / "work"
RESULTS_DIR = WORK_ROOT / "results"
WORK_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(WORK_ROOT / "hf_cache")
print("WORK_DIR:", WORK_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

In [ ]:
#@title 2. 拉取研究分支代码并安装固定版本依赖
%%bash
set -euo pipefail
REPO_DIR="/content/EM-FloatFormer"
BRANCH="research/stage4-math-and-realdata"
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone --depth 1 -b "$BRANCH" https://github.com/20171806011/EM-FloatFormer.git "$REPO_DIR"
else
  git -C "$REPO_DIR" fetch origin "$BRANCH" --depth 1
  git -C "$REPO_DIR" checkout "$BRANCH"
  git -C "$REPO_DIR" reset --hard "origin/$BRANCH"
fi
cd "$REPO_DIR"
git log -n 2 --oneline
pip install -q "transformers>=5.24.0" accelerate bitsandbytes huggingface_hub safetensors sentencepiece protobuf "numpy==2.3.5" "scipy==1.17.0" h5py
pip install -q flash-linear-attention || echo "[INFO] flash-linear-attention optional build skipped; pure PyTorch/CUDA fallback active." 

In [ ]:
#@title 3. 从 GitHub Release (stage4-4b-colab-artifacts) 拉取并 SHA-256 校验 Stage 4 检查点与 Real-SDR 双缓存
import subprocess, sys
%cd /content/EM-FloatFormer
if "/content/EM-FloatFormer" not in sys.path:
    sys.path.insert(0, "/content/EM-FloatFormer")

cmd = [
    sys.executable, "-B", "research/stage4_real_and_surgical/colab_stage6_runner.py",
    "--stage", "download_artifacts",
    "--work-dir", str(WORK_DIR),
    "--results-dir", str(RESULTS_DIR),
]
subprocess.run(cmd, check=True)

In [ ]:
#@title 4. 运行 Stage 6A：Qwen3.5-4B Real-SDR + W2 Lie+Schur Soup + CN-SFT Real-SDR + Stage 6 Active EIG
import subprocess, sys
%cd /content/EM-FloatFormer

SEED = 202611
STEPS_STAGE6 = 240

cmd = [
    sys.executable, "-B", "research/stage4_real_and_surgical/colab_stage6_runner.py",
    "--stage", "stage6_4b",
    "--work-dir", str(WORK_DIR),
    "--results-dir", str(RESULTS_DIR),
    "--seed", str(SEED),
    "--steps-stage6", str(STEPS_STAGE6),
]
subprocess.run(cmd, check=True)

In [ ]:
#@title 5. 运行 Stage 6B：Qwen3.5-9B CN-SFT + Real-SDR + Stage 6 Active EIG (需要 >=20 GiB VRAM)
import subprocess, sys
%cd /content/EM-FloatFormer
if "/content/EM-FloatFormer" not in sys.path:
    sys.path.insert(0, "/content/EM-FloatFormer")
from research.stage4_real_and_surgical.colab_stage6_scaling import detect_colab_gpu_profile

prof = detect_colab_gpu_profile()
FORCE_RUN_9B = False
if not prof["models"]["9b"].get("supported", False) and not FORCE_RUN_9B:
    print(f"[SKIP stage6_9b] Current GPU ({prof['gpu_name']}, {prof['vram_gib']} GiB): {prof['models']['9b'].get('reason')}")
else:
    cmd = [
        sys.executable, "-B", "research/stage4_real_and_surgical/colab_stage6_runner.py",
        "--stage", "stage6_9b",
        "--work-dir", str(WORK_DIR),
        "--results-dir", str(RESULTS_DIR),
        "--seed", "202611",
        "--steps-stage6", "240",
        "--g1-blocks-scaled", "32",
    ]
    subprocess.run(cmd, check=True)

In [ ]:
#@title 6. 运行 Stage 6C：Qwen3.8-27B Mixed-NF4 (W4 定理 2 审计) CN-SFT + Real-SDR + Stage 6 Active EIG (需要 >=35 GiB VRAM)
import subprocess, sys
%cd /content/EM-FloatFormer
if "/content/EM-FloatFormer" not in sys.path:
    sys.path.insert(0, "/content/EM-FloatFormer")
from research.stage4_real_and_surgical.colab_stage6_scaling import detect_colab_gpu_profile

prof = detect_colab_gpu_profile()
FORCE_RUN_27B = False
if not prof["models"]["27b"].get("supported", False) and not FORCE_RUN_27B:
    print(f"[SKIP stage6_27b] Current GPU ({prof['gpu_name']}, {prof['vram_gib']} GiB): {prof['models']['27b'].get('reason')}")
else:
    cmd = [
        sys.executable, "-B", "research/stage4_real_and_surgical/colab_stage6_runner.py",
        "--stage", "stage6_27b",
        "--work-dir", str(WORK_DIR),
        "--results-dir", str(RESULTS_DIR),
        "--seed", "202611",
        "--steps-stage6", "240",
        "--g1-blocks-scaled", "32",
    ]
    subprocess.run(cmd, check=True)

In [ ]:
#@title 7. 打包所有 JSON 结果报告为 ZIP 并触发本地下载
import json, zipfile

zip_path = WORK_ROOT / "stage6_results_bundle.zip"
json_files = sorted(RESULTS_DIR.glob("*.json"))
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for jf in json_files:
        zf.write(jf, arcname=jf.name)

summary = [{"file": jf.name, "bytes": jf.stat().st_size} for jf in json_files]
print("Bundled ZIP:", zip_path, f"({zip_path.stat().st_size} bytes)")
print(json.dumps(summary, indent=2, ensure_ascii=False))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print("Browser auto-download skipped:", exc)